# Preprocess OM files on solar energy production

- PV household data from [openmeter platform (OM)](https://appstore.logarithmo.de/app/openmeterplatform/v1/demo/page-datenuebersicht?lang=DE)


In [1]:
import pandas as pd
import numpy as np

In [2]:
example_file = "openmeter_Zeitreihendaten_Energiemessungen_5dda585a-a6a8-4938-b68c-3435de3d3c1a.csv"
data_path = "../data/omp_data/"

In [3]:
def preprocess_om_sensor_file(datafile, path):
    """_summary_

    Args:
        datafile (string): filename of data csv from openmeter platform
        path (string): location of datafile

    Returns:
        df, dict: dataframe with measured data + metadata dictionary
    """
    data = []
    header_data = True
    metadata_dict = {}
    with open(path+datafile, 'r') as infile:
        # iterate through file
        for line in infile:
            if line.strip() != "" and not line.strip() == ",":
                # identify start of measurement data
                if line.startswith("Zeitstempel"):
                    header_data = False
                # collect metadata
                elif header_data:
                    info = line.strip().split(",")
                    if info[1] == "":
                        info[1] = np.nan
                    metadata_dict[info[0]] = info[1]
                # store measurement data
                else:
                    data.append(line.strip().split(","))

    # convert to proper dataframe
    df = pd.DataFrame(data)
    df.columns = ["timestamp", "measurement"]

    # correct metadata on measurement date range
    if metadata_dict["measures_from"] != df.iloc[1].timestamp.split(" ")[0] or\
    metadata_dict["measures_to"] != df.iloc[1].timestamp.split(" ")[1]:
        print("Correcting measurement time range")
        print("  Start data: %s > %s" % (metadata_dict["measures_from"], df.iloc[0].timestamp.split(" ")[0]))
        print("  End data:   %s > %s" % (metadata_dict["measures_to"], df.iloc[-1].timestamp.split(" ")[0]))
        metadata_dict["measures_from"] = df.iloc[0].timestamp.split(" ")[0]
        metadata_dict["measures_to"] = df.iloc[-1].timestamp.split(" ")[0]
    df2 = pd.DataFrame(metadata_dict.items())
    df2.columns = ["info", "value"]

    # save preprocessed dataframes to file
    sensor = datafile.split("_")[-1].split(".")[0]
    df.to_csv(path+"openmeter_sensor_"+sensor+"_data.csv", index=False)
    df2.to_csv(path+"openmeter_sensor_"+sensor+"_metadata.csv", index=False)

    return df, df2

In [4]:
if __name__ == "__main__":
    # read data and metadata from file
    data_df, metadata_df = preprocess_om_sensor_file(example_file, data_path)

    # inspect dataframe
    print(data_df.head())

Correcting measurement time range
  Start data: 2022-07-14 > 2021-01-28
  End data:   2023-07-05 > 2023-07-19
             timestamp   measurement
0  2021-01-28 00:00:00  3633.2240277
1  2021-01-28 00:15:00  3633.2240277
2  2021-01-28 00:30:00  3633.2240277
3  2021-01-28 00:45:00  3633.2240277
4  2021-01-28 01:00:00  3633.2240277


In [5]:
if __name__ == "__main__":
    print("Corresponding metadata:")
    print(metadata_df)

Corresponding metadata:
                    info                                 value
0                     id  b50ba775-9f69-491d-b901-3272c4b77255
1          measures_from                            2021-01-28
2            measures_to                            2023-07-19
3   measurement_category                             Erzeugung
4       measurement_unit                                   kWh
5       measurement_type                            Wirkarbeit
6               category                                Privat
7                  usage                              Haushalt
8           usage_detail                                   NaN
9                country                           Deutschland
10         federal_state                                Bayern
11                  city                               Buchloe
12             post_code                                 86808
13                  area                                    70
14     construction_year       